# Rastreabilidade Assistencial no SUS via Data Linkage
### Auditoria da População de Pacientes (jul de 2024 à jun de 2025) entre RNDS e SIA/SIH utilizando CPF/CNS como Identificador Único”

Avaliação da integridade e completude do fluxo de dados assistenciais no SUS, verificando a sobreposição e as exclusões (pacientes) entre os sistemas de Regulação (RNDS) e Faturamento (SIA/SIH) para o ano de 2024. A análise utiliza o CPF ou CNS como chave de linkage, alinhando-se aos princípios da Portaria 6.656/2025.

### Objetivo Principal:
Demonstrar a porcentagem de sobreposição de pacientes e procedimentos entre as bases. Especificamente:
•	Comprovar a rastreabilidade: Determinar a proporção de pacientes concluídos na RNDS (Regulação) que efetivamente aparecem nas bases de faturamento (SIA/SIH).
•	Identificar gargalos/inconsistências: Determinar a proporção de pacientes faturados (SIA/SIH) que não passaram ou não tiveram registro de conclusão na RNDS, evidenciando falhas no registro de Regulação Assistencial.
•	Validar a RNDS: Comprovar estatisticamente que os dados da RNDS, apesar de iniciais, já fornecem uma base válida para estudos de fluxo assistencial e tempo de espera.


### Metodologia de Ciência de Dados e Estatística (Foco em Linkage):
#### ETAPA 1: Analise e tratamento.
Devido ao volume de dados os arquivos estão em Parquet, veja o 'convert_csv_parquet.ipynb' 


In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import os

parquet_path_sia = "base/SIA.parquet" # Use barras normais
num_linhas_limite = 1000

print(f"Tentando carregar as primeiras {num_linhas_limite} linhas de: {parquet_path_sia}")

if os.path.exists(parquet_path_sia):
    try:
        # 1. Abre o arquivo Parquet como um ParquetFile (objeto PyArrow)
        parquet_file = pq.ParquetFile(parquet_path_sia)

        # 2. Lê apenas as primeiras N linhas do arquivo.
        # Ele lê os Row Groups necessários até atingir o limite de linhas.
        tabela_limitada = parquet_file.read_table(num_rows=num_linhas_limite)

        # 3. Converte a tabela PyArrow limitada para um DataFrame do Pandas
        df_sia_amostra = tabela_limitada.to_pandas()

        print("✅ Leitura de amostra concluída com sucesso!")
        print(f"DataFrame Carregado com {len(df_sia_amostra)} linhas.")

        # ----------------------------------------------------
        # Começo do Tratamento de Dados (Exemplo)
        # ----------------------------------------------------
        
        # Exemplo: Remover linhas onde 'CPF_PAC' é NULL/vazio
        print("\nIniciando tratamento de dados na amostra...")
        
        # 1. Tratar valores nulos (NULL)
        # Cria uma máscara para remover linhas onde 'CPF_PAC' é nulo
        # Se você quer considerar strings vazias (''), adicione df_sia_amostra['CPF_PAC'] != ''
        df_sia_tratado = df_sia_amostra.dropna(subset=['CPF_PAC'])
        
        linhas_removidas = len(df_sia_amostra) - len(df_sia_tratado)
        
        print(f"Total de linhas antes do tratamento: {len(df_sia_amostra)}")
        print(f"Linhas removidas (CPF_PAC nulo): {linhas_removidas}")
        print(f"Total de linhas após o tratamento: {len(df_sia_tratado)}")
        
        print("\n--- Primeiras linhas após tratamento ---")
        print(df_sia_tratado.head())

    except Exception as e:
        print(f"❌ Erro ao carregar/tratar a amostra: {e}")
else:
    print(f"❌ ERRO: Arquivo '{parquet_path_sia}' não encontrado.")

Tentando carregar as primeiras 1000 linhas de: base/SIA.parquet
❌ Erro ao carregar/tratar a amostra: 'ParquetFile' object has no attribute 'read_table'
